In [5]:
# @title
from pathlib import Path
import json
import zipfile
import pandas as pd

FILE_PATH = "/content/accounting_timeseries.csv"
OUTPUT_DIR = Path("/content/accounting_timeseries_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 200_000
SAMPLE_PER_CHUNK = 50
RECENT_YEAR_FROM = 2021

dtype = {
    "inn": "string",
    "report_year": "Int64",
    "period_name": "string",
    "calendar_year": "Int64",
    "report_type": "string",
    "form_id": "string",
    "power": "Float64",
    "code": "string",
    "is_subrow": "boolean",
    "row_name": "string",
    "column": "string",
    "value": "Float64",
    "source": "string",
}

total_rows = 0
missing_counts = None
unique_inn = set()
report_year_counts = {}
calendar_year_counts = {}
form_counts = {}
power_counts = {}
source_counts = {}
samples = []
recent_samples = []
dictionary_parts = []

min_report_year = None
max_report_year = None
min_calendar_year = None
max_calendar_year = None

def detect_csv_encoding(file_path: str) -> str:
    """
    Проверяет наиболее вероятные кодировки по небольшому фрагменту файла.
    """
    with open(file_path, "rb") as file:
        raw = file.read(2_000_000)

    for encoding in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            raw.decode(encoding)
            return encoding
        except UnicodeDecodeError:
            continue

    return "cp1251"


detected_encoding = detect_csv_encoding(FILE_PATH)
print("Определённая кодировка:", detected_encoding)

reader = pd.read_csv(
    FILE_PATH,
    dtype=dtype,
    chunksize=CHUNK_SIZE,
    low_memory=False,
    encoding=detected_encoding,
    encoding_errors="replace",
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)

    if missing_counts is None:
        missing_counts = chunk.isna().sum()
    else:
        missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0)

    unique_inn.update(chunk["inn"].dropna().astype(str).unique())

    for column_name, accumulator in [
        ("report_year", report_year_counts),
        ("calendar_year", calendar_year_counts),
        ("form_id", form_counts),
        ("power", power_counts),
        ("source", source_counts),
    ]:
        counts = chunk[column_name].value_counts(dropna=False)
        for key, count in counts.items():
            key = "<NA>" if pd.isna(key) else str(key)
            accumulator[key] = accumulator.get(key, 0) + int(count)

    report_year = chunk["report_year"].dropna()
    if not report_year.empty:
        current_min = int(report_year.min())
        current_max = int(report_year.max())
        min_report_year = (
            current_min if min_report_year is None
            else min(min_report_year, current_min)
        )
        max_report_year = (
            current_max if max_report_year is None
            else max(max_report_year, current_max)
        )

    calendar_year = chunk["calendar_year"].dropna()
    if not calendar_year.empty:
        current_min = int(calendar_year.min())
        current_max = int(calendar_year.max())
        min_calendar_year = (
            current_min if min_calendar_year is None
            else min(min_calendar_year, current_min)
        )
        max_calendar_year = (
            current_max if max_calendar_year is None
            else max(max_calendar_year, current_max)
        )

    sample_size = min(SAMPLE_PER_CHUNK, len(chunk))
    samples.append(chunk.sample(sample_size, random_state=42 + chunk_number))

    recent = chunk.loc[
        chunk["calendar_year"].fillna(0) >= RECENT_YEAR_FROM
    ]
    if not recent.empty:
        recent_samples.append(
            recent.sample(
                min(SAMPLE_PER_CHUNK, len(recent)),
                random_state=1000 + chunk_number,
            )
        )

    dictionary_parts.append(
        chunk[
            ["report_type", "form_id", "code", "is_subrow", "row_name"]
        ].drop_duplicates()
    )

    if chunk_number % 10 == 0:
        print(
            f"Обработано частей: {chunk_number}; "
            f"строк: {total_rows:,}; "
            f"уникальных ИНН: {len(unique_inn):,}"
        )

random_sample = pd.concat(samples, ignore_index=True)
random_sample = random_sample.sample(
    min(5000, len(random_sample)),
    random_state=42,
)

if recent_samples:
    recent_sample = pd.concat(recent_samples, ignore_index=True)
    recent_sample = recent_sample.sample(
        min(5000, len(recent_sample)),
        random_state=42,
    )
else:
    recent_sample = pd.DataFrame(columns=random_sample.columns)

indicator_dictionary = (
    pd.concat(dictionary_parts, ignore_index=True)
    .drop_duplicates()
    .sort_values(
        ["form_id", "code", "row_name"],
        na_position="last",
    )
)

missing_report = pd.DataFrame({
    "column": missing_counts.index,
    "missing_count": missing_counts.astype("int64").values,
})
missing_report["missing_share"] = (
    missing_report["missing_count"] / total_rows
)

def counts_to_frame(data, column_name):
    return pd.DataFrame(
        [{column_name: key, "rows": value} for key, value in data.items()]
    ).sort_values("rows", ascending=False)

random_sample.to_csv(
    OUTPUT_DIR / "01_random_sample.csv",
    index=False,
    encoding="utf-8-sig",
)

recent_sample.to_csv(
    OUTPUT_DIR / "02_recent_years_sample.csv",
    index=False,
    encoding="utf-8-sig",
)

indicator_dictionary.to_csv(
    OUTPUT_DIR / "03_indicator_dictionary.csv",
    index=False,
    encoding="utf-8-sig",
)

missing_report.to_csv(
    OUTPUT_DIR / "04_missingness.csv",
    index=False,
    encoding="utf-8-sig",
)

counts_to_frame(
    report_year_counts,
    "report_year",
).to_csv(
    OUTPUT_DIR / "05_report_year_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

counts_to_frame(
    calendar_year_counts,
    "calendar_year",
).to_csv(
    OUTPUT_DIR / "06_calendar_year_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

counts_to_frame(
    form_counts,
    "form_id",
).to_csv(
    OUTPUT_DIR / "07_form_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

counts_to_frame(
    power_counts,
    "power",
).to_csv(
    OUTPUT_DIR / "08_power_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

counts_to_frame(
    source_counts,
    "source",
).to_csv(
    OUTPUT_DIR / "09_source_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

summary = {
    "file_name": Path(FILE_PATH).name,
    "file_size_bytes": Path(FILE_PATH).stat().st_size,
    "rows": total_rows,
    "unique_inn": len(unique_inn),
    "columns": list(dtype),
    "min_report_year": min_report_year,
    "max_report_year": max_report_year,
    "min_calendar_year": min_calendar_year,
    "max_calendar_year": max_calendar_year,
    "unique_indicator_definitions": int(len(indicator_dictionary)),
}

with open(
    OUTPUT_DIR / "10_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

zip_path = Path("/content/accounting_timeseries_audit.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(OUTPUT_DIR.iterdir()):
        archive.write(file_path, arcname=file_path.name)

print("\nПРОВЕРКА ЗАВЕРШЕНА")
print(f"Строк: {total_rows:,}")
print(f"Уникальных ИНН: {len(unique_inn):,}")
print(
    f"Годы отчетов: {min_report_year}–{max_report_year}"
)
print(
    f"Календарные годы: {min_calendar_year}–{max_calendar_year}"
)
print(f"Определений показателей: {len(indicator_dictionary):,}")
print(f"Готовый пакет: {zip_path}")

Определённая кодировка: utf-8-sig
Обработано частей: 10; строк: 2,000,000; уникальных ИНН: 28,258
Обработано частей: 20; строк: 4,000,000; уникальных ИНН: 53,159

ПРОВЕРКА ЗАВЕРШЕНА
Строк: 4,589,027
Уникальных ИНН: 60,726
Годы отчетов: 2004–2025
Календарные годы: 2002–2025
Определений показателей: 1,004
Готовый пакет: /content/accounting_timeseries_audit.zip
